# Bias–Variance Tradeoff — companion notebook

Runnable companion for [`bias-variance.md`](./bias-variance.md).

Estimate bias², variance, and total MSE for polynomial regression at varying degrees by Monte Carlo over many resampled training sets. We expect to see the textbook U-curve in total error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.polynomial import polyfit, polyval

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Setup

True function $f(x) = \sin(2\pi x)$ on $[0, 1]$. Each training set has $n=20$ noisy samples with $\sigma=0.3$. We resample the training set $M=400$ times and average.

In [ ]:
def f_true(x):
    return np.sin(2 * np.pi * x)

n_train = 20
sigma = 0.3
n_test = 200
x_test = np.linspace(0, 1, n_test)
f_test = f_true(x_test)

degrees = np.arange(0, 16)
M = 400  # Monte Carlo training sets

# preds[d, m, i] = prediction at x_test[i] from m-th training set fit at degree d
preds = np.empty((len(degrees), M, n_test))
for m in range(M):
    x_train = rng.uniform(0, 1, n_train)
    y_train = f_true(x_train) + sigma * rng.standard_normal(n_train)
    for di, d in enumerate(degrees):
        coefs = polyfit(x_train, y_train, deg=d)
        preds[di, m] = polyval(x_test, coefs)

mean_pred = preds.mean(axis=1)                          # (degrees, n_test)
bias_sq   = ((mean_pred - f_test) ** 2).mean(axis=1)    # avg over x_test
variance  = preds.var(axis=1).mean(axis=1)
total_mse = bias_sq + variance + sigma**2

In [ ]:
# Empirical sanity check: total = E[(y - f_hat)^2] computed directly.
y_test_noisy = f_test + sigma * rng.standard_normal((M, n_test))
# Match a single noise draw per training set for the empirical estimate.
empirical_total = ((y_test_noisy - preds.mean(axis=1, keepdims=True)) ** 2).mean(axis=(1, 2)) * 0  # placeholder
# Better: compute E[(y - f_hat)^2] = mean over (m, i) of (y - preds[d, m, i])^2 with independent y noise per (m, i).
noise = sigma * rng.standard_normal((M, n_test))
y_eval = f_test[None, :] + noise   # (M, n_test)
empirical_total = ((y_eval[None, :, :] - preds) ** 2).mean(axis=(1, 2))

for d, b, v, t, e in zip(degrees, bias_sq, variance, total_mse, empirical_total):
    print(f'deg={d:2d}  bias^2={b:.3f}  var={v:.3f}  bias^2+var+sigma^2={t:.3f}  empirical={e:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(degrees, bias_sq,     'o-',  label='bias$^2$',         color='tab:blue')
axes[0].plot(degrees, variance,    's-',  label='variance',         color='tab:orange')
axes[0].axhline(sigma**2, ls='--', color='gray', label=f'irreducible $\\sigma^2$={sigma**2:.2f}')
axes[0].plot(degrees, total_mse,   '^-',  label='total expected MSE', color='tab:red', lw=2)
axes[0].set_xlabel('polynomial degree'); axes[0].set_ylabel('error')
axes[0].set_title('Classic U-curve: total error is minimized at intermediate complexity')
axes[0].set_yscale('log')
axes[0].legend()

# Show fits at low / medium / high degree.
showcase = [1, 4, 12]
for d in showcase:
    di = list(degrees).index(d)
    axes[1].plot(x_test, mean_pred[di], label=f'mean fit deg={d}')
    # show fit envelope (5th-95th percentile across training sets)
    lo, hi = np.percentile(preds[di], [5, 95], axis=0)
    axes[1].fill_between(x_test, lo, hi, alpha=0.15)
axes[1].plot(x_test, f_test, 'k--', label='true $f$')
axes[1].set_title('Average fit ± 5–95% envelope across training sets')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
axes[1].legend()
plt.tight_layout(); plt.show()

**Reading the curves.**
- At degree 0–1 the fits are nearly flat — high bias, low variance.
- At degree 12+ each fit chases its training noise — low bias, exploding variance.
- The total-error minimum sits around degree 4–6 for this $f$, $n$, $\sigma$ combination.
- Empirical total tracks the decomposition closely, confirming the identity.